# CSV Resampling and trimming
This code will trimm the original, 5GB, CSV file down and resample to a minute

In [9]:
import pandas as pd

# Load the data
data = pd.read_csv("SSS-August.csv", encoding='UTF-16', parse_dates=['time'])
data.head()
data['time'] = pd.to_datetime(data['time'],format='ISO8601') # The time format gave me problems, so it is specified here.
start_date = '2024-08-05 00:00:00'
end_date = '2024-08-6 23:59:59'
start_date = pd.to_datetime(start_date).tz_localize('UTC')
end_date = pd.to_datetime(end_date).tz_localize('UTC')


# The following time stamps are the week of interest, not used in testing:
#start_date = '2024-08-05 00:00:00'
#end_date = '2024-08-11 23:59:59'

# We trimm the data to work with down
trimmed_data_1 = data[(data['time'] >= start_date) & (data['time'] <= end_date)]
trimmed_data = trimmed_data_1.copy()


After trimming the data, the data is sampled for each unique combination of device_id+measurement_id in 1 min intervals.

In [10]:
resampled_dfs = []

# First we iterate over each unique combination of device_id and 
for (device_id, measurement_id), group in trimmed_data.groupby(['device_id', 'measurement_id']):
    # Set time as the index for resampling
    group = group.set_index('time')
    # Resample to 1-minute intervals and aggregate the values
    resampled = group['value_float'].resample('1min').mean()  # Replace 'mean' with desired aggregation if needed
    # Reset the index
    resampled = resampled.reset_index()
    # Add the device_id and measurement_id back to the resampled DataFrame
    resampled['device_id'] = device_id
    resampled['measurement_id'] = measurement_id
    resampled['application_id'] = group['application_id'].iloc[0]  # First value of application_id
    resampled['name'] = group['name'].iloc[0]  # First value of name
    # Append to the list of resampled DataFrames
    resampled_dfs.append(resampled)

# Concatenate all the resampled DataFrames
resampled_df = pd.concat(resampled_dfs, ignore_index=True)

# Save the resampled data to a new CSV
resampled_df.to_csv("SSS-August-small-resampled-v3.csv", index=False, encoding='utf-16')

print("Resampled the data with original file structure")

Resampled the data with original file structure


### Full dataset

In [7]:
import pandas as pd

# Load the data
data = pd.read_csv("SSS-August.csv", encoding='UTF-16', parse_dates=['time'])
data.head()
data['time'] = pd.to_datetime(data['time'],format='ISO8601') # The time format gave me problems, so it is specified here.
start_date = '2024-08-01 00:00:00'
end_date = '2024-08-31 23:59:59'
start_date = pd.to_datetime(start_date).tz_localize('UTC')
end_date = pd.to_datetime(end_date).tz_localize('UTC')


# The following time stamps are the week of interest, not used in testing:
#start_date = '2024-08-05 00:00:00'
#end_date = '2024-08-11 23:59:59'

# We trimm the data to work with down
trimmed_data_1 = data[(data['time'] >= start_date) & (data['time'] <= end_date)]
trimmed_data = trimmed_data_1.copy()


In [9]:
resampled_dfs = []

# First we iterate over each unique combination of device_id and 
for (device_id, measurement_id), group in trimmed_data.groupby(['device_id', 'measurement_id']):
    # Set time as the index for resampling
    group = group.set_index('time')
    # Resample to 1-minute intervals and aggregate the values
    resampled = group['value_float'].resample('1min').mean()  # Replace 'mean' with desired aggregation if needed
    # Reset the index
    resampled = resampled.reset_index()
    # Add the device_id and measurement_id back to the resampled DataFrame
    resampled['device_id'] = device_id
    resampled['measurement_id'] = measurement_id
    resampled['application_id'] = group['application_id'].iloc[0]  # First value of application_id
    resampled['name'] = group['name'].iloc[0]  # First value of name
    # Append to the list of resampled DataFrames
    resampled_dfs.append(resampled)

# Concatenate all the resampled DataFrames
resampled_df = pd.concat(resampled_dfs, ignore_index=True)

# Save the resampled data to a new CSV
resampled_df.to_csv("SSS-August-full-set.csv", index=False, encoding='utf-16')

print("Resampled the data with original file structure")

Resampled the data with original file structure
